# Parte 1: clasificación de sentimiento con ML clásico

**Curso:** Aprendizaje de Máquina 2026-20, Universidad de los Andes
**Grupo:** _TODO: nombre del grupo en Kaggle_
**Integrantes:** _TODO: nombres y códigos_

## Objetivo
Predecir el sentimiento **global** (`negativo`, `neutral`, `positivo`) de reseñas de productos en español. La métrica de Kaggle es **accuracy**.

## Restricciones de la Parte 1 (si se incumplen, la nota es 0)
- Solo clasificadores de **scikit-learn**.
- Nada de redes neuronales, transformers ni embeddings preentrenados.
- Representación del texto: bolsa de palabras, TF-IDF, n-gramas.
- Resultados **replicables** a partir de este notebook y del modelo guardado con `joblib`.

## Estructura del notebook
1. Configuración
2. Carga de datos
3. Análisis exploratorio y corrección de los datos
4. Partición de los datos y esquema de validación
5. Representación del texto y modelos base
6. Más allá de los n-gramas: dónde está la opinión
7. Ajuste de hiperparámetros
8. Evaluación y análisis de errores
9. Modelo final
10. Bitácora de envíos y conclusiones
11. Declaración de uso de IA generativa

## 1. Configuración

In [ ]:
import re
import unicodedata
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.svm import LinearSVC

SEED = 42  # usar en todo lo que tenga azar para que el resultado sea replicable
np.random.seed(SEED)

DATA_DIR = Path("data")
SUB_DIR = Path("submissions")  # archivos de envío a Kaggle
MODEL_DIR = Path("models")     # modelo final guardado con joblib
SUB_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

LABELS = ["negativo", "neutral", "positivo"]

print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__)

## 2. Carga de datos

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
eval_df = pd.read_csv(DATA_DIR / "eval.csv")  # SOLO para predecir al final, nunca para entrenar ni ajustar

X, y = train["text"], train["label"]
print(train.shape, eval_df.shape)
train.head()

## 3. Análisis exploratorio

En esta sección solo se **observan** los datos para saber qué hay. No se limpia ni se modifica nada: `train` y `eval_df` quedan intactos.

### 3.1 Estructura y calidad básica
¿Cuántas filas y columnas hay, de qué tipo son, hay nulos, textos vacíos o repetidos?

In [ ]:
for nombre, df in [("train", train), ("eval", eval_df)]:
    print(f"--- {nombre}: {df.shape[0]} filas, {df.shape[1]} columnas")
    print(df.dtypes.to_string())
    print("Nulos por columna:", df.isna().sum().to_dict())
    print("Textos vacíos:", (df["text"].str.strip() == "").sum())
    print("Ids repetidos:", df["id"].duplicated().sum(), "| Textos repetidos:", df["text"].duplicated().sum())
    print()

print("Textos de eval que también aparecen en train:", eval_df["text"].isin(train["text"]).sum())

### 3.2 Distribución de clases
¿Qué valores toma `label` y qué tan balanceadas están las clases? Sirve para saber qué accuracy tendría un modelo que siempre predice la clase más frecuente.

In [ ]:
conteo = y.value_counts()
resumen_clases = pd.DataFrame({"reseñas": conteo, "porcentaje": (conteo / len(y) * 100).round(1)})
print("Valores distintos de label:", sorted(y.unique()))
display(resumen_clases)

ax = conteo.reindex(LABELS).plot.bar(rot=0, title="Reseñas por clase")
ax.bar_label(ax.containers[0])
plt.show()

print("Accuracy de predecir siempre la clase más frecuente:", round(conteo.max() / len(y), 3))

### 3.3 Longitud de las reseñas
¿Qué tan largas son (caracteres, palabras, oraciones)? ¿Cambia la longitud según la clase? ¿Se parecen `train` y `eval`?

In [ ]:
def medidas(textos):
    """Tabla con la longitud de cada texto. No modifica los datos originales."""
    return pd.DataFrame({
        "caracteres": textos.str.len(),
        "palabras": textos.str.split().str.len(),
        "oraciones": textos.str.count(r"[.!?]+"),
    })

largos = medidas(X).assign(label=y)
print("Longitud en train (todas las clases):")
display(largos.describe().round(1))
print("Promedios por clase:")
display(largos.groupby("label").mean().round(1))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["caracteres", "palabras", "oraciones"]):
    sns.boxplot(data=largos, x="label", y=col, order=LABELS, ax=ax)
    ax.set_title(f"{col.capitalize()} por clase")
plt.tight_layout(); plt.show()

print("Promedios train vs. eval:")
display(pd.DataFrame({"train": medidas(X).mean(), "eval": medidas(eval_df["text"]).mean()}).round(1))

### 3.4 Leer ejemplos
Leer reseñas reales de cada clase es la parte más importante del EDA: aquí se ve cómo están escritas y qué las hace positivas, negativas o neutrales.

In [ ]:
for label in LABELS:
    print(f"===== {label}")
    for t in train.loc[y == label, "text"].sample(5, random_state=SEED):
        print("-", t)
    print()

In [ ]:
# Las reseñas más cortas y las más largas
orden = largos["palabras"].sort_values()
for titulo, idx in [("MÁS CORTAS", orden.index[:5]), ("MÁS LARGAS", orden.index[-3:])]:
    print(f"===== {titulo}")
    for i in idx:
        print(f"[{y[i]} | {largos.loc[i, 'palabras']} palabras] {X[i]}")
    print()

### 3.5 Caracteres y ruido en el texto
¿Qué tipo de caracteres aparecen? Tildes, mayúsculas, números, signos, emojis, enlaces. Esto muestra qué habría que normalizar más adelante (aquí solo se mide, no se corrige).

In [ ]:
patrones = {
    "con tildes o ñ": r"[áéíóúüñÁÉÍÓÚÜÑ]",
    "con números": r"\d",
    "con signos ¡ o ¿": r"[¡¿]",
    "con signos ! o ?": r"[!?]",
    "con palabras en MAYÚSCULAS": r"\b[A-ZÁÉÍÓÚÑ]{3,}\b",
    "con enlaces o @": r"https?://|www\.|@",
    "con caracteres fuera del español": r"[^\x00-\x7FáéíóúüñÁÉÍÓÚÜÑ¡¿]",
}
ruido = pd.DataFrame({
    nombre: {
        "train (%)": X.str.contains(p).mean() * 100,
        **{f"{l} (%)": X[y == l].str.contains(p).mean() * 100 for l in LABELS},
        "eval (%)": eval_df["text"].str.contains(p).mean() * 100,
    }
    for nombre, p in patrones.items()
}).T.round(1)
print("Porcentaje de reseñas que contienen cada tipo de carácter:")
display(ruido)

# Inventario de caracteres que no son letras, números ni espacios
from collections import Counter
simbolos = Counter(c for t in X for c in t if not c.isalnum() and not c.isspace())
print("Símbolos presentes en train:", dict(simbolos.most_common()))

### 3.6 Vocabulario
¿Cuántas palabras distintas hay, cuáles son las más frecuentes y cuáles aparecen mucho más en una clase que en las otras? ¿Qué tanto vocabulario de `eval` no está en `train`?

In [ ]:
# Solo se cuenta para explorar: en minúsculas, sin quitar tildes ni stopwords
contador = CountVectorizer(lowercase=True, binary=True)  # binary=True: cuenta en cuántas reseñas aparece cada palabra
M = contador.fit_transform(X)
palabras = contador.get_feature_names_out()
print("Palabras distintas en train:", len(palabras))

# Porcentaje de reseñas de cada clase que contienen cada palabra
frec = pd.DataFrame({l: np.asarray(M[(y == l).values].mean(axis=0)).ravel() * 100 for l in LABELS}, index=palabras)
frec["todas"] = np.asarray(M.mean(axis=0)).ravel() * 100

print("\nPalabras más frecuentes (% de reseñas que las contienen):")
display(frec.sort_values("todas", ascending=False).head(20).round(1))

for l in LABELS:
    otras = [o for o in LABELS if o != l]
    diferencia = frec[l] - frec[otras].max(axis=1)
    print(f"\nPalabras mucho más frecuentes en '{l}' que en las otras clases:")
    display(frec.loc[diferencia.sort_values(ascending=False).head(15).index, LABELS].round(1))

vocab_eval = set(CountVectorizer(lowercase=True).fit(eval_df["text"]).get_feature_names_out())
nuevas = vocab_eval - set(palabras)
print(f"\nPalabras de eval que no están en train: {len(nuevas)} de {len(vocab_eval)} ({len(nuevas) / len(vocab_eval):.1%})")
print("Ejemplos:", sorted(nuevas)[:30])

**Conclusiones del EDA** _(TODO: completar con lo que observen)_
- Balance de clases: ...
- Estructura típica de una reseña (relleno, opinión, conector, opinión final): ...
- Los conectores aparecen casi igual en positivas y negativas, así que lo que importa es **qué viene después**: ...
- Ruido (reseñas sin tildes, errores de tipeo): ...

**Hipótesis:** el sentimiento global lo determina la última cláusula de opinión. Se pone a prueba en la sección 6.

### 3.7 Corrección de los datos
Con lo que mostró el EDA se corrige **solo** lo que de verdad aparece en los datos. Los textos originales no se tocan: el resultado queda en una columna nueva, `text_limpio`.

| Hallazgo del EDA | Corrección |
|---|---|
| El 31 % de las reseñas no tiene tildes ni ñ ("llego", "tamano") y el resto sí | Quitar tildes y ñ en todas, para que "llegó" y "llego" sean la misma palabra |
| Hay mayúsculas al inicio de oración y en siglas ("USB", "AAA") | Pasar todo a minúsculas |
| 3 reseñas de `train` tienen la codificación dañada ("reseÃ±a", "llegA3") | Reparar esos pares de caracteres ("reseña", "llegó") |
| 1 reseña más tiene un guion invisible (`\xad`) suelto en medio de una palabra | Eliminarlo |

**Lo que no se corrige, y por qué:**
- **Filas:** no se elimina ninguna, porque no hay nulos, textos vacíos ni repetidos.
- **Etiquetas:** solo existen los tres valores esperados.
- **Palabras vacías ("no", "pero", "aun así"):** se conservan, porque cambian el sentido de la opinión.
- **Puntuación:** se conserva, porque marca dónde termina cada oración.
- **Números:** se conservan; aparecen sobre todo en las reseñas neutrales, así que ayudan a distinguirlas.
- **Errores de tipeo ("trs dias", "autonoia"):** no se corrigen, porque no hay una regla segura para adivinar la palabra correcta.

La corrección es una regla fija (no aprende nada de los datos), así que aplicarla también a `eval` no usa `eval` para entrenar.

In [ ]:
# Pares que deja una codificación dañada (UTF-8 leído como Latin-1) -> letra correcta
MOJIBAKE = {"Ã¡": "á", "Ã©": "é", "Ã\xad": "í", "Ã³": "ó", "Ãº": "ú", "Ã±": "ñ"}
# Variante en la que además la Ã perdió su tilde: "llegA3" -> "llegó"
MOJIBAKE_DEGRADADO = {"A¡": "á", "A©": "é", "A\xad": "í", "A3": "ó", "A³": "ó", "Aº": "ú", "A±": "ñ"}
PATRON_DEGRADADO = re.compile(r"(?<=[a-zñ])A[¡©\xad3³º±]")  # solo dentro de una palabra, para no tocar "AA" o "AAA"

def reparar_par(coincidencia):
    return MOJIBAKE_DEGRADADO[coincidencia.group()]

def corregir_texto(texto):
    """Devuelve una copia corregida del texto. No modifica el original."""
    for malo, bueno in MOJIBAKE.items():                 # 1. reparar la codificación dañada
        texto = texto.replace(malo, bueno)
    texto = PATRON_DEGRADADO.sub(reparar_par, texto)
    texto = texto.replace("\xad", "")                    # 2. quitar guiones invisibles
    texto = texto.lower()                                # 3. minúsculas
    texto = unicodedata.normalize("NFKD", texto)         # 4. quitar tildes y ñ
    return "".join(c for c in texto if not unicodedata.combining(c))

train["text_limpio"] = train["text"].map(corregir_texto)
eval_df["text_limpio"] = eval_df["text"].map(corregir_texto)

print("Filas en train:", len(train), "| en eval:", len(eval_df), "(no se eliminó ninguna)")
train[["text", "text_limpio", "label"]].head(3)

**Comprobación.** Las mismas medidas del EDA, antes y después de corregir, y ejemplos de cómo cambia el texto.

In [ ]:
PATRON_DANADO = r"Ã|(?<=[a-zñ])A[¡©\xad3³º±]|\xad"

def resumen(textos):
    return {
        "reseñas con tildes o ñ": textos.str.contains(r"[áéíóúüñÁÉÍÓÚÜÑ]").sum(),
        "reseñas con mayúsculas": textos.str.contains(r"[A-ZÁÉÍÓÚÑ]").sum(),
        "reseñas con codificación dañada o guion invisible": textos.str.contains(PATRON_DANADO).sum(),
        "palabras distintas": len(CountVectorizer().fit(textos).vocabulary_),
        "textos repetidos": textos.duplicated().sum(),
        "textos vacíos": (textos.str.strip() == "").sum(),
    }

display(pd.DataFrame({
    "train antes": resumen(train["text"]),
    "train después": resumen(train["text_limpio"]),
    "eval antes": resumen(eval_df["text"]),
    "eval después": resumen(eval_df["text_limpio"]),
}))

# Todos los caracteres distintos que quedan después de corregir
print("Caracteres que quedan en train:", "".join(sorted(set("".join(train["text_limpio"])))))
print("Caracteres que quedan en eval: ", "".join(sorted(set("".join(eval_df["text_limpio"])))))

# Palabras de eval que no están en train, antes y después
for col in ["text", "text_limpio"]:
    vocab_train = set(CountVectorizer().fit(train[col]).vocabulary_)
    vocab_eval = set(CountVectorizer().fit(eval_df[col]).vocabulary_)
    nuevas = vocab_eval - vocab_train
    print(f"{col}: {len(nuevas)} de {len(vocab_eval)} palabras de eval no están en train ({len(nuevas) / len(vocab_eval):.1%})")

# Ejemplos: una reseña normal y las que tenían la codificación dañada
danadas = train.index[train["text"].str.contains(PATRON_DANADO)]
for i in [0, *danadas]:
    print(f"\n[{i}] ANTES:   {train.loc[i, 'text'][:160]}")
    print(f"[{i}] DESPUÉS: {train.loc[i, 'text_limpio'][:160]}")

## 4. Partición de los datos y esquema de validación
`eval.csv` no tiene etiquetas, así que no sirve para medir el modelo. Hay que construir la evaluación con `train.csv`.

### 4.1 Conjunto de prueba propio
> **Qué hacer aquí:** apartar una parte de `train.csv` (por ejemplo el 20 %) como prueba, de forma estratificada y con la semilla fija. Ese conjunto no se vuelve a tocar hasta la sección 9.

**Decisión** _(TODO: tamaño de la prueba y por qué; comprobar que las tres clases conservan su proporción)_

### 4.2 Validación cruzada dentro de entrenamiento
> **Qué hacer aquí:** definir la validación cruzada estratificada que se usará para comparar modelos y elegir hiperparámetros, y una función que evalúe cualquier pipeline y guarde el resultado en una tabla de experimentos.
>
> Todo lo que aprende de los datos (vocabulario, pesos IDF) debe ir dentro de un `Pipeline`, para que se ajuste solo con la parte de entrenamiento de cada pliegue.

**Decisión** _(TODO: número de pliegues, por qué estratificada, y por qué no se elige el modelo mirando la prueba)_

## 5. Representación del texto y modelos base
En esta sección se recorre el camino visto en clase: de la representación más simple a la más completa, midiendo cada cambio con validación cruzada.

### 5.1 Línea base trivial
> **Qué hacer aquí:** medir un clasificador que siempre predice la clase más frecuente. Todo lo demás se compara contra ese número.

### 5.2 Bolsa de palabras contra TF-IDF
> **Qué hacer aquí:** el mismo clasificador (por ejemplo regresión logística) con las dos representaciones. Probar también `min_df` y `sublinear_tf`.

**Resultado** _(TODO: ¿cuál representa mejor estas reseñas y por qué?)_

### 5.3 N-gramas
> **Qué hacer aquí:** agregar bigramas y trigramas (`ngram_range`). Son la herramienta del curso para recuperar parte del orden: "no me gustó", "aun así". Medir cuánto crece el vocabulario y cuánto mejora el resultado.

**Resultado** _(TODO: ¿cuánto ayudan los n-gramas? ¿alcanzan para capturar qué opinión va al final?)_

### 5.4 Palabras vacías y reducción a la raíz
> **Qué hacer aquí:** comparar con y sin palabras vacías, y con y sin stemming en español.
>
> Cuidado: en análisis de sentimiento, quitar "no" cambia el significado ("no me gustó" quedaría igual que "me gustó"). Un filtro que descarte palabras de dos letras o menos también borraría "no".

**Decisión** _(TODO: qué se conserva, qué se quita y por qué, con los números de cada prueba)_

### 5.5 Clasificadores
> **Qué hacer aquí:** con la mejor representación, comparar los clasificadores del curso: regresión logística, Naive Bayes, árbol de decisión, KNN, Random Forest y boosting. Dejar todo en una sola tabla.
>
> Antes de ejecutar, anotar qué esperan de cada uno con una matriz dispersa de miles de columnas (por ejemplo, qué le pasa a KNN con tantas dimensiones).

**Conclusiones de los modelos base** _(TODO: ¿qué combinación funciona mejor? ¿en qué valor se estanca la bolsa de palabras y por qué?)_

### 5.6 Función para generar envíos a Kaggle
> **Qué hacer aquí:** una función que entrene con los datos etiquetados, prediga `eval.csv`, valide el formato (`id,answer`, 3.000 filas, etiquetas válidas) y guarde el archivo en `submissions/`. Usarla para el primer envío.

## 6. Más allá de los n-gramas: dónde está la opinión
El EDA sugiere que la opinión final de la reseña es la que decide la etiqueta. Esta sección prueba si se puede aprovechar eso sin salir de TF-IDF y n-gramas: aplicándolos a **partes** del texto.

> ⚠️ Antes de usar esto en el modelo final, confirmar con la profesora o el monitor que segmentar el texto antes de vectorizar está permitido. _(TODO: anotar la respuesta y la fecha)_
>
> Toda función que vaya dentro del pipeline se define con `def` en este notebook (nunca con `lambda`), para que el modelo se pueda guardar y volver a cargar.

### 6.1 ¿Qué parte del texto tiene la información?
> **Qué hacer aquí:** entrenar el mismo modelo usando solo la primera oración, solo la última, y solo lo que va después del último conector contrastivo. Comparar.

**Resultado** _(TODO: ¿se confirma que la información está al final?)_

### 6.2 Combinar el texto completo con sus partes
> **Qué hacer aquí:** unir las vistas del texto (completo, última oración, última cláusula) y agregar una a la vez, midiendo cuánto aporta cada una.

**Resultado** _(TODO: ¿qué vistas aportan y cuáles no? Enviar a Kaggle solo las mejoras reales)_

## 7. Ajuste de hiperparámetros
> **Qué hacer aquí:** búsqueda de hiperparámetros sobre el mejor pipeline (en malla, aleatoria o por reducción sucesiva), con validación cruzada dentro de entrenamiento.
>
> Recordar que en `LogisticRegression` más regularización es **bajar** `C`. Anotar cuántos entrenamientos implica la búsqueda (combinaciones × pliegues).

**Resultado** _(TODO: mejores hiperparámetros, cuánto mejoró frente al valor por defecto y si hay señales de sobreajuste)_

## 8. Evaluación y análisis de errores
Todo en esta sección se hace con predicciones de validación cruzada sobre entrenamiento. La prueba sigue sin tocarse.

### 8.1 Matriz de confusión y métricas por clase
> **Qué hacer aquí:** matriz de confusión y precisión, recall y F1 de cada clase. La exactitud sola no dice qué clases se confunden entre sí.

**Cómo se lee** _(TODO: ¿qué clases se confunden? ¿hay alguna que ya esté resuelta?)_

### 8.2 Qué aprendió el modelo
> **Qué hacer aquí:** mostrar los términos con más peso hacia cada clase (por ejemplo, los coeficientes de la regresión logística) y revisar si tienen sentido.

**Cómo se lee** _(TODO: ¿los términos más importantes coinciden con lo que se vio en el EDA?)_

### 8.3 Lectura de errores
> **Qué hacer aquí:** leer al menos 30 reseñas mal clasificadas y agruparlas por tipo de error.

**Tipos de error encontrados** _(TODO)_

| Tipo de error | Ejemplo | Cambio propuesto |
|---|---|---|
| ... | ... | ... |

Con los cambios propuestos se vuelve a la sección 5 o 6 y se mide de nuevo.

## 9. Modelo final
### 9.1 Evaluación en la prueba (una sola vez)
> **Qué hacer aquí:** evaluar el modelo elegido en el conjunto de prueba apartado en 4.1. Solo se hace una vez, cuando ya no se va a cambiar nada.

**Resultado** _(TODO: comparar con la validación cruzada. Si la prueba da un poco menos, ¿por qué es esperable?)_

### 9.2 Reentrenamiento, envío y guardado
> **Qué hacer aquí:** reentrenar el modelo final con todo `train.csv`, generar el envío y guardar el modelo con `joblib` en `models/`.
>
> El modelo entregado debe ser el del **mejor score público** del grupo en Kaggle.

### 9.3 Prueba de replicabilidad
> **Qué hacer aquí:** cargar el modelo guardado y comprobar que sus predicciones sobre `eval.csv` son idénticas a las del envío.

## 10. Bitácora de envíos y conclusiones
Mínimo **5 envíos distintos** a Kaggle.

| # | Fecha | Modelo / cambio | Exactitud en validación cruzada | Score público Kaggle |
|---|-------|-----------------|---------------------------------|----------------------|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |
| 4 | | | | |
| 5 | | | | |

Línea base del curso en Kaggle: _TODO_

### Conclusiones finales _(TODO)_
- Mejor modelo y por qué: ...
- Qué aportó cada decisión (con números de la tabla de experimentos): ...
- Lo que este modelo no puede hacer (orden, negación, sinónimos, palabras nuevas) y por qué eso motiva la Parte 2: ...

## 11. Declaración de uso de IA generativa
El curso pide, para el proyecto, declarar el uso de IA generativa y anexar los prompts.

| Pregunta | Respuesta |
|---|---|
| ¿Qué herramienta se usó? (nombre y versión) | _TODO_ |
| ¿Para qué se usó? | _TODO_ |
| ¿Qué parte del trabajo resultó de ella? | _TODO_ |
| ¿Cómo se verificó lo que produjo? | _TODO_ |

Anexo de prompts: _TODO (archivo o enlace)_